## DMP Preference learning

In [ ]:
# Load the same from Thermal_comfort example 
%load_ext autoreload
%autoreload 2

#isualize the plots in the notebook itself
%matplotlib inline 

import sys,os
sys.path.append('../')   
import numpy as np
from model.exactPreference import  exactPreference
# from model.LuceJND import  LuceJND
# from model.erroneousPreference import  erroneousPreference
# from model.gaussianNoisePreferenceFull import  gaussianNoisePreference
from kernel import RBF
from kernel.jaxrbf import RBF as JRBF
from utility import  paramz
# for plotting
import matplotlib.pyplot as plt
import arviz as az

### 1D case 

In [ ]:
# True utility function

bounds=[[10,35]]
def fun(x,noise=0.1):
    return 0.3*np.exp(-(x-16)**2/15)+0.4*np.exp(-(x-23)**2/10)+0.2*np.exp(-(x-31)**2/4)+noise
Xpred=np.linspace(bounds[0][0],bounds[0][1],201)[:,None] # Prediction points

# The best x value 
x_best = Xpred[np.argmax(fun(Xpred[:,0]))]
print("x_best", x_best)
 
plt.plot(Xpred[:,0],fun(Xpred[:,0]))
plt.xlabel("x (Celsius)",fontsize=16)
plt.ylabel("u(x)",fontsize=16)
plt.grid()
plt.show()
#plt.savefig("figures/ThermalHouse.pdf")

In [ ]:
# Create pair-wise preference dataset 

n=251 # number of objects (now it's one digit number)
X=np.linspace(bounds[0][0],bounds[0][1],n)[:,None] # Does this need to be rounded? -> Try without
print("X:", X)  #Getting the integer values between 11-35 in this case
#print("length of X:", len(X))

def generate_human_feedback(query_points_idx, X): # TODO: Implement more robust index conversion
    pairs = []
    m = len(query_points_idx)
    
    # Generate all pairwise comparisons
    for idx_i in range(m):
        for idx_j in range(idx_i + 1, m):  # Only upper triangle to avoid duplicates
            i = int(query_points_idx[idx_i])
            j = int(query_points_idx[idx_j])
            print("i, j", i, j)
            
            # Compare based on true utility
            if fun(X[i]) > fun(X[j]):
                pairs.append([i, j])  # i is preferred to j
            else:
                pairs.append([j, i])  # j is preferred to i
    
    return pairs # This returns the pairs in index

query_indices = np.random.permutation(n) # Randomly order n numbers of query indices
print("query incides:", query_indices)

Pairs = generate_human_feedback(query_indices[:5], X) # Generate feedback for first 5 query indices 

# Pairs, indices = np.unique(Pairs, axis=0, return_index=True)#remove any duplicate pair
print(Pairs) 
print("Preference pairs (temperature values):", np.asarray([(X[i][0], X[j][0]) for i, j in Pairs]))  # Print the actual temperature values for the pairs


In [ ]:
# Learn the utility function 

# Data dictionary
data = {}
data["Pairs"] = Pairs# Convert to indices
data["X"] = X


#kernel parameter dictionary
params={'lengthscale': {'value':1.5*np.ones(data["X"].shape[1],float), 
                        'range':np.vstack([[0.1, 20.0]]*data["X"].shape[1]),
                        'transform': paramz.logexp()},
        'variance': {'value':np.array([1.0]), 
                'range':np.vstack([[1.0, 1.0001]]),
                'transform': paramz.logexp()}#variance not used in exactPreference
          }
#define inference method for the hyperparameters
inf_method="laplace"
if inf_method=="advi":
    Kernel = JRBF
elif inf_method=="laplace":
    Kernel = RBF

#define preference model 
model = exactPreference(data,Kernel,params,inf_method=inf_method) # Initialize exact preference model
#compute hyperparameters
#model.optimize_hyperparams(num_restarts=2) we do not optimise the hyperparameters
print(model.params)
#sample from posterior
model.sample(nsamples=30000, tune=2000) # Sample from posterior
#predicted samples
predictions = model.predict(Xpred) # This returns (200, 30000) -> Posterior samples for 201 temperature points.


#compute the lower and upper credible intervals
credib_int = az.hdi(predictions,0.95)

 97%|█████████▋| 31096/32000 [00:05<00:00, 5633.73it/s]

In [ ]:
#plot the latent function mean and credible interval

print(Xpred.shape)
print(credib_int.shape)

plt.plot(Xpred[:,0],credib_int[:,1],color='C0', linestyle=':', label='95% CI')
plt.plot(Xpred[:,0],credib_int[:,0],color='C0', linestyle=':')
plt.plot(Xpred[:,0],np.mean(predictions,axis=1), label='mean',color='C0')
plt.xlabel("x (Celsius)",fontsize=16)
plt.ylabel("u(x)",fontsize=16)
plt.legend()
plt.grid()
plt.show()

In [ ]:
# Get the next query point
from scipy.stats import norm

# --- TODO: Maybe try different approach to compute y_best --- (this function is currently not used)
def compute_y_best(current_query_indices, mu, X, Xpred, method='observed'):

    if method == 'predicted':
        return np.max(mu)
    
    elif method == 'observed':
        pass 
        return # y_best 
# --- TODO

def expected_improvement(x, mu, sigma, y_best):
    x = np.atleast_2d(x)
    #print(x)
    with np.errstate(divide='ignore'): # ignore warning of divide by sigma = 0 
        z = (mu - y_best) / sigma
        #print(z) 
        ei = (mu - y_best) * norm.cdf(z) + sigma * norm.pdf(z)
        ei[sigma == 0.0] = 0.0 # expected improvement is zero if sigma is 0 
    return ei #.ravel()

mu = np.mean(predictions, axis=1) 
sigma = np.std(predictions, axis=1) 
y_best = np.max(mu) # Is this correct? - I guess so.

#print(X)
EI = expected_improvement(Xpred, mu, sigma, y_best) # This will return the expected improvement of each X point 

# Now I find x that has max EI 
idx_next = np.argmax(EI)
x_next = Xpred[idx_next, 0]
ei_max = np.max(EI)

print("Next query point:", x_next)
print("EI value:", ei_max)

In [ ]:
# Get new query points (now added to the next cell, keep this for maybe debugging later)
debug = False

if debug is True: 
    new_query_indices = query_indices[5:9]
    # new_query_points = X[new_query_indices].flatten()
    # new_query_points = np.hstack([x_next, new_query_points]).tolist()
    # print(new_query_points)
    
    new_pairs = generate_human_feedback(new_query_indices, X)
    print(new_pairs)
    
    all_pairs = Pairs + new_pairs  # Combine old and new pairs
    print(f"Total pairs: {len(all_pairs)} (was {len(Pairs)}, added {len(new_pairs)})")
    
    # Create updated model with all pairs
    data = {}
    data["Pairs"] = all_pairs
    data["X"] = X
    
    # Reinitialize model with accumulated pairs
    model = exactPreference(data, Kernel, params, inf_method=inf_method)
    
    # Sample from updated posterior
    model.sample(nsamples=20000, tune=1000)
    
    # Make predictions
    predictions = model.predict(Xpred)
    
    # Compute EI
    mu = np.mean(predictions, axis=1)
    sigma = np.std(predictions, axis=1)
    y_best = np.max(mu)
    EI = expected_improvement(Xpred, mu, sigma, y_best)
    
    # Find next query point
    idx_next = np.argmax(EI)
    x_next = float(Xpred[idx_next, 0])
    
    # Plot updated results
    credib_int_updated = az.hdi(predictions.T, 0.95)
    plt.plot(Xpred[:,0], credib_int_updated[:,1], color='C0', linestyle=':', label='95% CI (updated)')
    plt.plot(Xpred[:,0], credib_int_updated[:,0], color='C0', linestyle=':')
    plt.plot(Xpred[:,0], np.mean(predictions, axis=1), label='mean (updated)', color='C0')
    plt.axvline(x_next, color='red', linestyle='--', alpha=0.7, label=f'Next query: {x_next:.1f}')
        
    plt.xlabel("x (Celsius)", fontsize=16)
    plt.ylabel("u(x)", fontsize=16)
    plt.legend()
    plt.grid()
else:
    pass

In [ ]:
# a = np.arange(0, 10, 1)
# print(a)

# idx = np.where(a == 5)[0][0]
# print(idx)
# print(type(idx))

In [ ]:
# Iterate for 4 more times

iteration_data = []

# Active learning loop
for iteration in range(4):  # 4 more iterations

    # Get new data
    start_idx = 5 + iteration * 4  # +5 for first round
    print("x_next", x_next)
    new_query_indices = query_indices[start_idx:start_idx+4]
    x_next_idx = np.where(X[:, 0] == np.round(x_next,1))[0]  # Find the index of x_next in X
    print("x_next_idx", x_next_idx)
    new_query_indices = np.hstack([new_query_indices,x_next_idx])
    # new_query_points = X[query_indices[start_idx:start_idx+4]].flatten().tolist()
    # new_query_points = [x_next] + new_query_points
    print("new_query_indices", new_query_indices)
    new_pairs = generate_human_feedback(new_query_indices, X)
    print("new pairs", new_pairs)
    
    # Accumulate pairs
    Pairs = Pairs + new_pairs
    print("Pairs", Pairs) 
    
    # Retrain model with ALL pairs
    data = {"Pairs": Pairs, "X": X}
    model = exactPreference(data, Kernel, params, inf_method=inf_method)
    model.sample(nsamples=30000, tune=2000)  # Consistent sampling
    predictions = model.predict(Xpred)

    # Compute EI
    mu = np.mean(predictions, axis=1)
    sigma = np.std(predictions, axis=1)
    y_best = np.max(mu)
    EI = expected_improvement(Xpred, mu, sigma, y_best)
    
    # Find next query point
    idx_next = np.argmax(EI)
    x_next = float(Xpred[idx_next, 0])
    
    print(f"Iteration {iteration+1}: {len(Pairs)} total pairs, next query: {x_next:.2f}")

    # Save iteration data for plotting later
    iteration_data.append({
        'iteration': iteration+2,
        'pairs': Pairs.copy(),
        'predictions': predictions.copy(),
        'x_next': x_next,
        'EI': EI.copy(),
        'num_pairs': len(Pairs)
    })

    # Print separation for clarity
    print("========================================")

# Create figure with subplots for all iterations
fig, axes = plt.subplots(2, 2, figsize=(15, 12))
axes = axes.flatten()

# Plot each iteration
for idx, data_iter in enumerate(iteration_data):
    ax = axes[idx]
    
    preds = data_iter['predictions']
    credib_int = az.hdi(preds, 0.95)
    
    # Plot true utility function
    # ax.plot(Xpred[:,0], fun(Xpred[:,0]), 'k--', alpha=0.5, label='True utility', linewidth=2)
    
    # Plot credible interval
    ax.fill_between(Xpred[:,0], credib_int[:,0], cHej hur många L är det?redib_int[:,1], 
                     color='C0', alpha=0, label='95% CI')
    ax.plot(Xpred[:,0], credib_int[:,1], color='C0', linestyle=':', linewidth=1)
    ax.plot(Xpred[:,0], credib_int[:,0], color='C0', linestyle=':', linewidth=1)
    
    # Plot mean
    ax.plot(Xpred[:,0], np.mean(preds, axis=1), label='Posterior mean', color='C0', linewidth=2)
    
    # Plot next query point (if available)
    if 'x_next' in data_iter:
        ax.axvline(data_iter['x_next'], color='red', linestyle='--', 
                   alpha=0.7, label=f'Next query: {data_iter["x_next"]:.1f}')
    
    # Formatting
    ax.set_xlabel("x (Celsius)", fontsize=12)
    ax.set_ylabel("u(x)", fontsize=12)
    ax.set_title(f"Iteration {data_iter['iteration']}: {data_iter['num_pairs']} pairs", 
                 fontsize=14, fontweight='bold')
    ax.legend(loc='upper right', fontsize=10)
    ax.grid(True, alpha=0.3)

plt.tight_layout()
#plt.savefig('preference_learning_iterations.png', dpi=150, bbox_inches='tight')
plt.show()

Is this really more efficient than learning random 50 pairs -> Seems yes. Compare with Thermal_comfort_objective_copy. (But that may not be because of next_x, maybe because all the temparature is queries at least once???)

In [ ]:
# Warm-started alternative to the 4-iteration loop above.
# This keeps the accumulated preference data, but reuses the previous posterior sample
# as the starting point for the next constrained sampler run.

from scipy.optimize import linprog
from inference import slice_sampler

def _sample_exact_preference_warm_start(model, previous_samples=None, nsamples=30000, tune=500, disable=False):
    if model.inf_method == 'advi':
        from utility.paramz import DictVectorizer
        params_kernel, _ = DictVectorizer().fit_transform(model.params)
        params_kernel = np.exp(params_kernel)
    else:
        params_kernel = model.params
    
    Kxx = model.Kernel(model.X, model.X, params_kernel) + np.eye(model.X.shape[0]) * model.jitter
    A = model.PrefM.toarray()
    b = np.zeros((A.shape[0], 1))
    
    x0 = None
    if previous_samples is not None and previous_samples.size > 0:
        feasible_mask = np.all(A @ previous_samples >= -1e-9, axis=0)
        feasible_indices = np.flatnonzero(feasible_mask)
        if feasible_indices.size > 0:
            x0 = previous_samples[:, feasible_indices[0]][:, None]
    
    if x0 is None:
        res = linprog(
            np.zeros(A.shape[1]),
            A_ub=-A,
            b_ub=-b,
            bounds=[(-1e-4, 1.0)] * A.shape[1],
            method='highs'
        )
        if not res.success:
            raise RuntimeError('Could not find a feasible starting point for the sampler.')
        x0 = res.x[:, None]
    
    samples = slice_sampler.liness_step(
        x0,
        A,
        b,
        np.zeros(Kxx.shape[0]),
        Kxx,
        nsamples=nsamples,
        tune=tune,
        progress=1 - disable
    ).T
    model.samples = samples
    return model, samples

def _predict_and_choose_next(model, Xpred):
    predictions = model.predict(Xpred)
    mu = np.mean(predictions, axis=1)
    sigma = np.std(predictions, axis=1)
    y_best = np.max(mu)
    EI = expected_improvement(Xpred, mu, sigma, y_best)
    idx_next = np.argmax(EI)
    x_next = float(Xpred[idx_next, 0])
    return predictions, EI, x_next

# Initial fit on the first 5 preference queries. (for degugging)
# warm_pairs = generate_human_feedback(query_indices[:5], X)
# warm_data = {"Pairs": warm_pairs, "X": X}
# warm_model = exactPreference(warm_data, Kernel, params, inf_method=inf_method)
# warm_model, warm_samples = _sample_exact_preference_warm_start(
#     warm_model,
#     previous_samples=None,
#     nsamples=30000,
#     tune=2000,
#     disable=False
#  )
# predictions, EI, x_next = _predict_and_choose_next(warm_model, Xpred)

# Initiallization for warm-started iterations
warm_pairs = []
warm_samples = None
x_next = None
warm_iteration_data = []

# Param for querying 
n_first_queries = 5
n_warm_iterations = 4

for iteration in range(n_warm_iterations):
    
    if iteration == 0: # First iteration
    # First round: use 5 random queries (no x_next yet)
    new_query_indices = query_indices[:n_first_queries]
    tune_steps = 2000 # What was this for?

    else: # Subsequent iterations
        start_idx = n_first_queries + iteration * 4
        new_query_indices = query_indices[start_idx:start_idx + 4]
        x_next_idx = int(round(x_next) - 11)
        new_query_indices = np.hstack([new_query_indices, x_next_idx])
    
    new_pairs = generate_human_feedback(new_query_indices, X)
    warm_pairs = warm_pairs + new_pairs
    
    warm_data = {"Pairs": warm_pairs, "X": X}
    warm_model = exactPreference(warm_data, Kernel, params, inf_method=inf_method)
    warm_model, warm_samples = _sample_exact_preference_warm_start(
        warm_model,
        previous_samples=warm_samples,
        nsamples=30000,
        tune=500,
        disable=False
    )
    predictions, EI, x_next = _predict_and_choose_next(warm_model, Xpred)
    
    warm_iteration_data.append({
        'iteration': iteration + 2,
        'pairs': warm_pairs.copy(),
        'predictions': predictions.copy(),
        'x_next': x_next,
        'EI': EI.copy(),
        'num_pairs': len(warm_pairs)
    })
    print(f"Warm-start iteration {iteration + 1}: {len(warm_pairs)} total pairs, next query: {x_next:.2f}")

fig, axes = plt.subplots(2, 2, figsize=(15, 12))
axes = axes.flatten()

for idx, data_iter in enumerate(warm_iteration_data):
    ax = axes[idx]
    preds = data_iter['predictions']
    credib_int = az.hdi(preds, 0.95)
    
    ax.fill_between(Xpred[:, 0], credib_int[:, 0], credib_int[:, 1],
                    color='C0', alpha=0, label='95% CI')
    ax.plot(Xpred[:, 0], credib_int[:, 1], color='C0', linestyle=':', linewidth=1)
    ax.plot(Xpred[:, 0], credib_int[:, 0], color='C0', linestyle=':', linewidth=1)
    ax.plot(Xpred[:, 0], np.mean(preds, axis=1), label='Posterior mean', color='C0', linewidth=2)
    ax.axvline(data_iter['x_next'], color='red', linestyle='--', alpha=0.7, label=f'Next query: {data_iter["x_next"]:.1f}')
    
    ax.set_xlabel('x (Celsius)', fontsize=12)
    ax.set_ylabel('u(x)', fontsize=12)
    ax.set_title(f"Warm-start iteration {data_iter['iteration']}: {data_iter['num_pairs']} pairs", fontsize=14, fontweight='bold')
    ax.legend(loc='upper right', fontsize=10)
    ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

### 3D case 

Parameters: a_d, sigma_d, k 

In [ ]:
# True utility function for 3D case

# Define bounds for 3D parameters
bounds_3d = [[1, 10],      # a_d (x)
             [-1, 1],      # sigma_d (y)
             [0.01, 15]]   # k (z)

def fun_3d(theta, noise=0.0):
    """
    3D utility function for DMP parameters.
    """
    # Ensure theta is at least 2D for consistent indexing
    theta = np.atleast_2d(theta)
    
    a_d = theta[:, 0]      # amplitude: [1, 10]
    sigma_d = theta[:, 1]  # goal shift: [-1, 1]
    k = theta[:, 2]        # stiffness: [0.01, 15]
    
    # Center and scale
    a_c = a_d - 5.5        # Center at 5.5
    s_c = sigma_d          # Already centered at 0
    k_c = k - 7.5          # Center at 7.5

    # TODO: Adjust the true utility value 
    utility = (
        # Separate quadratic term 
        -0.05 * a_c**2 +
        -0.3 * s_c**2 +
        -0.02 * k_c**2 +
        
        # Interactive term 
        0.03 * a_c * k_c +    
        -0.1 * a_c * np.abs(s_c) +  
        0.05 * k_c * np.exp(-s_c**2) + 
        
        # Offset
        1.0
    )
    
    utility += noise * np.random.randn(*utility.shape)
    
    return utility

In [ ]:
# Visualize 3D utility function with 2D slices

from mpl_toolkits.mplot3d import Axes3D

# Generate random samples in the parameter space
n_samples = 1000
np.random.seed(42)  # For reproducibility
random_theta = np.random.uniform(
    [bounds_3d[0][0], bounds_3d[1][0], bounds_3d[2][0]],
    [bounds_3d[0][1], bounds_3d[1][1], bounds_3d[2][1]],
    size=(n_samples, 3)
)
random_u = fun_3d(random_theta)

# Highlight the best sampled point
optimal_idx = np.argmax(random_u)
optimal_theta = random_theta[optimal_idx]
optimal_utility = float(random_u[optimal_idx])

# Create figure
fig = plt.figure(figsize=(12, 9))
ax = fig.add_subplot(111, projection='3d')

# Scatter plot colored by utility value
scatter = ax.scatter(random_theta[:, 0], random_theta[:, 1], random_theta[:, 2],
                     c=random_u, cmap='viridis', s=30, alpha=0.6, 
                     edgecolors='none')

# Mark the global optimum with a star
ax.scatter(*optimal_theta, s=300, color= 'red', marker='*', label=f'Optimum (u={optimal_utility:.3f})', 
          zorder=10)

# Add colorbar
cbar = plt.colorbar(scatter, ax=ax, shrink=0.6, pad=0.1)
cbar.set_label('Utility u(θ)', fontsize=12, rotation=270, labelpad=20)

# Labels and title
ax.set_xlabel('a_d ', fontsize=12, labelpad=10)
ax.set_ylabel('σ_d ', fontsize=12, labelpad=10)
ax.set_zlabel('k ', fontsize=12, labelpad=10)
ax.set_title('3D Parameter Space (colored by utility value)', 
            fontsize=14, pad=20)

# Set axis limits
ax.set_xlim(bounds_3d[0])
ax.set_ylim(bounds_3d[1])
ax.set_zlim(bounds_3d[2])

# Legend
ax.legend(loc='upper left', fontsize=11)

# Grid
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('3d_utility_visualization.png', dpi=150, bbox_inches='tight')
plt.show()

print(f"\nVisualization with {n_samples} random samples")
print(f"Optimal parameters: a_d={optimal_theta[0]:.2f}, σ_d={optimal_theta[1]:.2f}, k={optimal_theta[2]:.2f}")
print(f"Optimal utility: {optimal_utility:.3f}")